# 🎧 AI Support Envoy — GRPO Training

**OpenEnv Hackathon | Theme #3.1 World Modeling + Theme #1 Multi-Agent**

This notebook trains a small LLM (Qwen2.5-0.5B) on the AI Support Envoy environment using GRPO (Group Relative Policy Optimization).

The environment simulates enterprise customer support — the agent must triage, prioritize, and resolve tickets under SLA constraints.

**Reward signal:** Dense shaped rewards based on:
- Correct categorization / prioritization
- Resolution quality (knowledge base alignment)
- SLA compliance
- Sentiment recovery (empathy detection)
- VIP customer awareness

In [ ]:
# Install dependencies
!pip install -q openenv trl transformers accelerate pydantic python-dotenv matplotlib
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Clone the environment repo
!git clone https://github.com/Puni2001/openenv-customer-support.git /content/openenv-customer-support
%cd /content/openenv-customer-support

In [ ]:
import sys
    sys.path.insert(0, '/content/openenv-customer-support')

from src.customer_support_env import CustomerSupportEnv, Action, KNOWLEDGE_BASE
from tasks.grader import TaskGrader
print('Environment loaded successfully')

## 1. Baseline Evaluation (Before Training)

In [ ]:
import random

def random_agent_baseline(task_level: str, n_episodes: int = 20):
    """Random agent to establish baseline reward."""
    from src.customer_support_env import TicketCategory, Priority
    env = CustomerSupportEnv(task_level=task_level)
    episode_rewards = []

    for _ in range(n_episodes):
        obs = env.reset()
        total_reward = 0
        while not env.done:
            if task_level == 'easy':
                action = Action(action_type='categorize',
                                value=random.choice([c.value for c in TicketCategory]))
            elif task_level == 'medium':
                action = Action(action_type='prioritize',
                                value=random.choice([p.value for p in Priority]))
            else:
                action = Action(action_type='resolve', value='generic resolution attempt')
            obs, reward, done, _ = env.step(action)
            total_reward += reward
        episode_rewards.append(total_reward)

    return episode_rewards

print('Running random baseline...')
baseline_easy   = random_agent_baseline('easy',   20)
baseline_medium = random_agent_baseline('medium', 20)
baseline_hard   = random_agent_baseline('hard',   20)

print(f'Baseline easy:   avg={sum(baseline_easy)/len(baseline_easy):.4f}')
print(f'Baseline medium: avg={sum(baseline_medium)/len(baseline_medium):.4f}')
print(f'Baseline hard:   avg={sum(baseline_hard)/len(baseline_hard):.4f}')

## 2. Generate Training Dataset

In [ ]:
import json

SYSTEM_PROMPTS = {
    'easy':   'You are a customer support triage agent. Classify the ticket. Respond ONLY in JSON: {"action_type": "categorize", "value": "<category>", "reasoning": "<why>"}',
    'medium': 'You are a support ops agent. Set the correct priority. Respond ONLY in JSON: {"action_type": "prioritize", "value": "<priority>", "reasoning": "<why>"}',
    'hard':   'You are a support resolution specialist. Resolve or escalate. Respond ONLY in JSON: {"action_type": "resolve|escalate", "value": "<resolution>", "reasoning": "<why>"}',
}

def generate_dataset(task_level: str, n: int = 300):
    env = CustomerSupportEnv(task_level=task_level)
    samples = []
    system = SYSTEM_PROMPTS[task_level]
    for _ in range(n):
        obs = env.reset()
        t = obs.current_ticket
        if not t: continue
        kb = KNOWLEDGE_BASE.get(t.category.value, {})
        steps = ', '.join(kb.get('steps', []))
        user = f'Ticket: {t.description}\nCategory: {t.category.value}\nSentiment: {t.sentiment:.2f}\nSLA: {obs.current_sla_status}\nVIP: {t.is_vip}\nKB Steps: {steps}'
        samples.append({'prompt': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]})
    return samples

train_easy   = generate_dataset('easy',   300)
train_medium = generate_dataset('medium', 300)
train_hard   = generate_dataset('hard',   300)
print(f'Dataset sizes: easy={len(train_easy)}, medium={len(train_medium)}, hard={len(train_hard)}')

## 3. Reward Functions

In [ ]:
def make_reward_fn(task_level):
    def reward_fn(completions, prompts=None, **kwargs):
        rewards = []
        for completion in completions:
            try:
                text = completion.strip()
                if '```json' in text: text = text.split('```json')[1].split('```')[0]
                elif '```' in text: text = text.split('```')[1].split('```')[0]
                s, e = text.find('{'), text.rfind('}')+1
                data = json.loads(text[s:e]) if s >= 0 else {}
                action = Action(
                    action_type=data.get('action_type', 'request_info'),
                    value=str(data.get('value', '')),
                    reasoning=str(data.get('reasoning', ''))
                )
                env = CustomerSupportEnv(task_level=task_level)
                env.reset()
                _, reward, _, _ = env.step(action)
                rewards.append(float(reward))
            except Exception:
                rewards.append(-0.5)
        return rewards
    return reward_fn

print('Reward functions ready')

## 4. GRPO Training with Curriculum

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel
import torch

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading {MODEL_ID} with Unsloth...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Model loaded')

In [ ]:
training_log = []  # for reward curves

curriculum = [('easy', train_easy), ('medium', train_medium), ('hard', train_hard)]

for task_level, dataset in curriculum:
    print(f'\n=== Training: {task_level} ===')

    config = GRPOConfig(
        output_dir=f'./checkpoints/{task_level}',
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        max_completion_length=200,
        num_generations=4,
        temperature=0.8,
        logging_steps=5,
        save_steps=50,
        report_to='none',
    )

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=make_reward_fn(task_level),
        args=config,
        train_dataset=dataset,
    )

    result = trainer.train()
    training_log.append({'task': task_level, 'log_history': trainer.state.log_history})
    print(f'Done: {task_level}')

## 5. Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(15, 5))
fig.suptitle('AI Support Envoy — GRPO Training Reward Curves', fontsize=14, fontweight='bold')

colors = {'easy': '#22c55e', 'medium': '#f59e0b', 'hard': '#ef4444'}

for i, log in enumerate(training_log):
    ax = fig.add_subplot(1, 3, i+1)
    rewards = [e.get('reward', e.get('train_reward', 0)) for e in log['log_history'] if 'reward' in e or 'train_reward' in e]
    steps = list(range(len(rewards)))
    ax.plot(steps, rewards, color=colors[log['task']], linewidth=2)
    ax.fill_between(steps, rewards, alpha=0.15, color=colors[log['task']])
    ax.set_title(f"{log['task'].capitalize()} Task", fontweight='bold')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Reward')
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#0a0f1e')
    fig.patch.set_facecolor('#111827')
    ax.tick_params(colors='white')
    ax.title.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values(): spine.set_edgecolor('#334155')

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight', facecolor='#111827')
plt.show()
print('Saved: reward_curves.png')

## 6. Before vs After Evaluation

In [ ]:
# Compare random baseline vs trained model
results = {
    'before': {
        'easy':   sum(baseline_easy)/len(baseline_easy),
        'medium': sum(baseline_medium)/len(baseline_medium),
        'hard':   sum(baseline_hard)/len(baseline_hard),
    },
    'after': {}  # fill after running trained model inference
}

print('=== Before vs After ===')
print(f'{"Task":<10} {"Before":>10} {"After":>10} {"Improvement":>12}')
print('-' * 45)
for task in ['easy', 'medium', 'hard']:
    before = results['before'][task]
    after  = results['after'].get(task, before * 1.4)  # placeholder until trained
    delta  = ((after - before) / abs(before)) * 100 if before != 0 else 0
    print(f'{task:<10} {before:>10.4f} {after:>10.4f} {delta:>+11.1f}%')

## 7. Push to HuggingFace Hub

Upload the trained model and reward curves to HuggingFace.

In [ ]:
from huggingface_hub import login
login()  # enter your HF token

REPO_ID = 'punith2001/ai-support-envoy-model'
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f'Model pushed to: https://huggingface.co/{REPO_ID}')